In [2]:


import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from imblearn.under_sampling import RandomUnderSampler
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.metrics import classification_report
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForCausalLM
import google.generativeai as genai
import os
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
import google.api_core.exceptions
from dotenv import load_dotenv

load_dotenv()



/home/cs/grad/islams32/dev/project/academic/satd/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [3]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [4]:
# train_df = pd.read_csv('../data/train.csv')
# train_df.head()
# train_x = []
# train_y = []
# for index, row in train_df.head(10).iterrows():
#     train_x.append(row['text'])
#     train_y.append(1)

# train_dataset = Dataset.from_dict({"text": train_x, "label":train_y})

# test_x = []
# test_y = []
# for index, row in pd.read_csv('../data/comment.csv').iterrows():
#     test_x.append(row['text'])
#     test_y.append(1 if row['is_td'] else 0)

# test_dataset = Dataset.from_dict({"text": test_x, "label":test_y})

# dataset = DatasetDict({'train': train_dataset, 'eval': test_dataset.shuffle(seed=42).select(range(10)), 'test': test_dataset.shuffle(seed=42).select(range(100))})

In [5]:
# df = pd.read_csv('../data/maldonado_corrected.csv')
# df['label'] = df['satd_orig'].apply(lambda x: 'yes' if x == 1 else 'no')
# df['text'] = df['comment_text']
# df = df[["text", "label"]]
#
# under_sampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
# X_resampled, y_resampled = under_sampler.fit_resample(df[['text']], df['label'])
#
# # Create balanced DataFrame
# df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
# dataset = Dataset.from_pandas(df).train_test_split(test_size=0.03, seed=42)
# dataset = dataset.remove_columns(['__index_level_0__'])
# dataset

In [6]:
df = pd.read_csv('../data/comment.csv')
df['label'] = df['is_td'].apply(lambda x: 'yes' if x == 1 else 'no')
df = df[["text", "label"]]

under_sampler = RandomUnderSampler(sampling_strategy=1, random_state=42)
X_resampled, y_resampled = under_sampler.fit_resample(df[['text']], df['label'])

# Create balanced DataFrame
df = pd.DataFrame({'text': X_resampled['text'], 'label': y_resampled})
dataset = Dataset.from_pandas(df).train_test_split(test_size=0.2, seed=42)
dataset = dataset.remove_columns(['__index_level_0__'])
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 804
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 202
    })
})

In [7]:
#Manually crafted Few Shots
curated_fs_df = pd.read_csv('../data/train.csv')
curated_fs_df['label'] = curated_fs_df['text'].apply(lambda x: 'yes')
manual_train_dataset = Dataset.from_pandas(curated_fs_df)
manual_train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 161
})

In [8]:
# import torch
# from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
# from datasets import Dataset
# from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# import numpy as np

# # Check if CUDA (GPU) is available
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# def tokenize_function(example, tokenizer):
#     return tokenizer(example["text"], padding=True, truncation=True, max_length=512)

# def compute_metrics(p):
#     """Compute metrics like accuracy, precision, recall, and F1 score."""
#     preds = np.argmax(p.predictions, axis=1)
#     acc = accuracy_score(p.label_ids, preds)
#     precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, preds, average='binary')
#     return {
#         "accuracy": acc,
#         "precision": precision,
#         "recall": recall,
#         "f1": f1
#     }

# def train_model(model_name, num_epochs, dataset):
#     """Train SATD detection model with a specified transformer model."""

#     GenerationConfig
#     tokenizer = AutoTokenizer.from_pretrained(model_name)
#     dataset = dataset.map(lambda x: tokenize_function(x, tokenizer), batched=True)

#     model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
#     model.to(device)  # Ensure the model is moved to the correct device (GPU or CPU)

#     training_args = TrainingArguments(
#         output_dir="./cache/results",
#         num_train_epochs=num_epochs,
#         eval_strategy="epoch",  # Updated from `evaluation_strategy` to `eval_strategy`
#         save_strategy="epoch",  # Matching evaluation_strategy
#         load_best_model_at_end=True,
#         logging_dir="./cache/logs",
#         logging_steps=10,             # Log every 10 steps
#         per_device_train_batch_size=2,  # Smaller batch size for few-shot
#         per_device_eval_batch_size=2,
#         warmup_steps=500,
#         weight_decay=0.01,
#         report_to="none"  # Prevent reporting to external services like WandB
#     )

#     trainer = Trainer(
#         model=model, 
#         args=training_args, 
#         train_dataset=dataset['train'], 
#         eval_dataset=dataset['eval'],
#         tokenizer=tokenizer,  # Updated to tokenizer=tokenizer, as this will be deprecated in future versions
#         compute_metrics=compute_metrics
#     )

#     trainer.train()
#     return model, tokenizer, dataset

# def predict(model, tokenizer, texts):
#     """Make predictions using the trained model."""
#     inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device)  # Move inputs to the same device
#     outputs = model(**inputs)
#     predictions = torch.argmax(outputs.logits, dim=1).cpu().numpy()  # Move outputs back to CPU for NumPy
#     return predictions

# # Example usage
# if __name__ == "__main__":
#     model_name = "roberta-base"  # Change this to switch models (e.g., "bert-base-uncased", "distilbert-base-uncased", etc.)
#     model, tokenizer, dataset = train_model(model_name,3, dataset=dataset)

#     # Test predictions on the test set only
#     test_samples = dataset['test']['text']
#     test_predictions = predict(model, tokenizer, test_samples)

#     test_labels = dataset['test']['label']
#     test_accuracy = accuracy_score(test_labels, test_predictions)

#     print(f"Test Accuracy: {test_accuracy}")

#     # Compute precision, recall, and F1 score for the test set
#     test_metrics = precision_recall_fscore_support(test_labels, test_predictions, average='binary')

#     print(f"Test Precision, Recall, F1: {test_metrics}")


In [9]:
class PromptTemplate:
    def __init__(self, name, definition, instruction, shot_template):
        self._name = name
        self._definition = definition
        self._instruction = instruction
        self._shot_template = shot_template

    @property
    def name(self):
        return self._name

    @property
    def definition(self):
        return self._definition

    @property
    def instruction(self):
        return self._instruction

    @property
    def shot_template(self):
        return self._shot_template

    def __repr__(self):
        return f"PromptTemplate(name={self.name}, description='{self.definition}', example='{self.shot_template}')"

In [10]:
class ModelConfig:
    def __init__(self, name: str, architecture: str, uri: str):
        self.name = name
        self.architecture = architecture
        self.uri = uri

    def __repr__(self):
        return f"ModelConfig(name='{self.name}', uri='{self.uri}')"

In [11]:
def get_token_length(tokenizer, text):
    if tokenizer:
        tokens = tokenizer(text, return_tensors='pt')
        return tokens['input_ids'].shape[1]
    else:
        return len(text)


def create_prompt(prompt_template, x, y, question, tokenizer):
    model_max_length = tokenizer.model_max_length if tokenizer else float('inf')
    initial_text = prompt_template.definition + " " + prompt_template.instruction
    allocated_tokens = len(x) + 5  # Formatting
    allocated_tokens += get_token_length(tokenizer, initial_text)
    question_prefix = question
    while len(question_prefix) > 0:
        target_sample_token_length = get_token_length(tokenizer,
                                                      prompt_template.shot_template.format(question_prefix, ''))
        if allocated_tokens + target_sample_token_length <= model_max_length:
            allocated_tokens += target_sample_token_length
            break
        else:
            question_prefix = question_prefix[: len(question_prefix) // 2]

    instances = []
    skipped = 0
    for index, [x, y] in enumerate(zip(x, y)):
        example_text = prompt_template.shot_template.format(x, y)
        example_token_length = get_token_length(tokenizer, example_text)
        if allocated_tokens + example_token_length < model_max_length:
            instances.append(prompt_template.shot_template.format(x, y))
            allocated_tokens += example_token_length
        else:
            skipped += 1
    if skipped > 0:
        print(f'Skipping {skipped} shots due to token limits')

    instances.append(prompt_template.shot_template.format(question_prefix, ''))
    prompt_text = initial_text + "\n" + "\n" + "\n\n".join(instances)
    prompt_token_length = get_token_length(tokenizer, prompt_text)
    if prompt_token_length > model_max_length:
        print(f'prompt length {prompt_token_length}')
    return prompt_text

In [12]:
def create_model_and_tokenizer(model_config: ModelConfig):
    is_hf_model = True
    uri = model_config.uri
    if 'gpt' in model_config.architecture.lower():
        model = AutoModelForCausalLM.from_pretrained(uri)
    elif "/bert" in model_config.architecture.lower() or "/codebert" in model_config.architecture.lower():
        model = AutoModelForSequenceClassification.from_pretrained(uri, num_labels=2)
    elif 'gemini' in model_config.architecture.lower():
        genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
        model = genai.GenerativeModel(uri)
        is_hf_model = False
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(uri)
    if is_hf_model:
        model.to(device)
        tokenizer = AutoTokenizer.from_pretrained(model_config.uri, use_fast=True)
    else:
        tokenizer = None
    return model, tokenizer


In [13]:
from enum import Enum


class FewShotSelectionStrategy(Enum):
    RANDOM = 'random'
    SIMILAR = 'similar'
    MANUAL_CRAFTED = 'manual_crafted'


In [14]:
import random


def pick_n_shot(x, y, index, st_similarity, n=0, strategy=None):
    if len(x) < n:
        raise Exception(f'only {len(x)} examples available for {n} shots')
    indexes = []
    if strategy == FewShotSelectionStrategy.RANDOM:
        indexes = random.sample(range(len(x)), n)
    elif strategy == FewShotSelectionStrategy.SIMILAR:
        _, top_n_indices = st_similarity[index].topk(n)
        indexes.extend(top_n_indices.tolist())
    elif strategy == FewShotSelectionStrategy.MANUAL_CRAFTED:
        indexes = [i for i in range(n)]
    shot_x = []
    shot_y = []
    for index in indexes:
        shot_x.append(x[index])
        shot_y.append(y[index])
    return shot_x, shot_y


In [15]:
sentence_transformer = SentenceTransformer('all-MiniLM-L6-v2')


In [16]:
def print_classification(test_x, test_y, y_pred):
    print("Incorrect Predictions:")
    for text, true, pred in zip(test_x, test_y, y_pred):
        if true != pred:
            print(f"✖ {text} (Label: {true}, Predicted: {pred})")


In [17]:
PROMPT_TEMPLATES = [PromptTemplate(
    name="No Keywords",
    definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments.",
    instruction="Assign the label of yes or no for each given source code comment and return the assigned label",
    shot_template="Comment: {}\nLabel: {}"
), PromptTemplate(
    name="MAT Keywords",
    definition="Self-admitted technical debt (SATD) are technical debt admitted by the developer through source code comments. SATD comments usually  contain specific keywords: TODO, FIXME, HACK, and XXX.",
    instruction="Assign the label of yes or no for each given source code comment.",
    shot_template="Comment: {}\nLabel: {}"
), PromptTemplate(
    name="Jitterbug Keywords",
    definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. SATD comments usually contain specific keywords: TODO, FIXME, HACK, and Workaround.",
    instruction="Assign the label of yes or no for each given source code comment.",
    shot_template="Comment: {}\nLabel: {}"
), PromptTemplate(
    name="GPT4 Keywords",
    definition="Self-admitted technical debt (SATD) is technical debt admitted by the developer through source code comments. SATD comments usually contain specific keywords: TODO, FIXME, HACK, XXX, NOTE, DEBT, REFACTOR, OPTIMIZE, TEMP, WORKAROUND, KLUDGE, REVIEW, NOFIX, PENDING, and BUG.",
    instruction="Assign the label of yes or no for each given source code comment.",
    shot_template="Comment: {}\nLabel: {}"
), PromptTemplate(
    name="TD",
    definition="Technical debt (TD) in code comment are comments that that indicates weak code or something need to be done. TD comments usually  contain specific keywords: TODO, FIXME, HACK, and XXX.",
    instruction="Assign the label of yes or no for each given source code comment.",
    shot_template="Comment: {}\nLabel: {}"
)]

In [18]:
FEW_SHOT_SIZES = [0, 1, 2, 3, 5, 10, 15, 20]
MODEL_CONFIGS = [
    ModelConfig(name="Flan T5 Small", architecture='flan-t5', uri="google/flan-t5-small"),
    ModelConfig(name="Flan T5 Base", architecture='flan-t5', uri="google/flan-t5-base"),
    ModelConfig(name="Flan T5 Large", architecture='flan-t5', uri="google/flan-t5-large"),
    ModelConfig(name="Flan T5 XL", architecture='flan-t5', uri="google/flan-t5-xl"),
    ModelConfig(name="BERT Base", architecture="bert", uri="google-bert/bert-base-uncased"),
    ModelConfig(name="CodeBERT Base", architecture="codebert", uri="microsoft/codebert-base"),
    ModelConfig(name="Facebook BART Base", architecture="bart", uri="facebook/bart-base"),
    ModelConfig(name="Gemini 2 Flash", architecture="gemini", uri="models/gemini-2.0-flash")

]
FEW_SHOT_STRATEGIES = [FewShotSelectionStrategy.MANUAL_CRAFTED, FewShotSelectionStrategy.RANDOM,
                       FewShotSelectionStrategy.SIMILAR]

In [19]:
def predict_with_prompt(model, tokenizer, prompt):
    if tokenizer:
        inputs = tokenizer(prompt, return_tensors='pt')
        inputs = {key: value.to(device) for key, value in inputs.items()}
        output = tokenizer.decode(
            model.generate(
                inputs["input_ids"],
                max_new_tokens=50
                # generation_config=GenerationConfig(max_new_tokens=5, do_sample=True, temperature=0.01)
            )[0],
            skip_special_tokens=True
        )
        return output.strip()
    else:
        return predict_with_gemini(model, prompt)


@retry(
    stop=stop_after_attempt(10),  # Stop after 5 retries
    wait=wait_exponential(multiplier=2, min=60, max=2 * 60),
    retry=retry_if_exception_type(google.api_core.exceptions.ResourceExhausted),  # Retry on rate limit errors
)
def predict_with_gemini(model, prompt):
    return model.generate_content(prompt).text.split()[-1]


In [20]:

def detect_satd(model_configs, few_shots, prompt_templates, few_shot_strategies, dataset):
    test_x = dataset["test"]["text"]
    test_y = dataset["test"]["label"]

    st_similarities = None

    for few_shot_size in few_shots:
        for model_config in model_configs:
            model, tokenizer = create_model_and_tokenizer(model_config)
            for prompt_template in prompt_templates:
                for few_shot_strategy in few_shot_strategies:
                    if few_shot_strategy == FewShotSelectionStrategy.MANUAL_CRAFTED:
                        train_x = manual_train_dataset["text"]
                        train_y = manual_train_dataset["label"]
                    else:
                        train_x = dataset["train"]["text"]
                        train_y = dataset["train"]["label"]
                    if few_shot_strategy == FewShotSelectionStrategy.SIMILAR and st_similarities is None:
                        st_similarities = cos_sim(sentence_transformer.encode(test_x),
                                                  sentence_transformer.encode(train_x))
                    y_pred = []
                    unknown_labels = []
                    for i, [text, label] in enumerate(zip(test_x, test_y)):
                        shot_x, shot_y = pick_n_shot(train_x, train_y, i, st_similarities, few_shot_size,
                                                     few_shot_strategy)
                        prompt = create_prompt(prompt_template, shot_x, shot_y, text, tokenizer)
                        pred = predict_with_prompt(model, tokenizer, prompt)
                        # if pred != label:
                        #     print(f'{pred} {label} Failing for {text}')
                        #     print(f'Prompt\n {prompt}')
                        if pred not in ['yes', 'no']:
                            unknown_labels.append(pred)
                            pred = 'no'
                        y_pred.append(pred)
                    prediction_df = pd.DataFrame()
                    prediction_df["label"] = test_y
                    prediction_df["pred_label"] = y_pred
                    prediction_df["text"] = test_x
                    run_config = f'{model_config.name} - {prompt_template.name} {few_shot_size} -  {few_shot_strategy}'
                    print(run_config)
                    print(classification_report(test_y, y_pred, zero_division=0, digits=3))
                    prediction_df.to_csv(f'./cache/{run_config}.csv')
                    print(f'Unknown classification count {len(unknown_labels)}')
                    print(f'Unknown classification  {unknown_labels}')
                    return test_y, y_pred
                    #print(y_pred)
                    #print(print_classification(test_x, test_y,y_pred))

In [21]:
# detect_satd(MODEL_CONFIGS[-1:], FEW_SHOT_SIZES[2:3], PROMPT_TEMPLATES[:1], FEW_SHOT_STRATEGIES[2:3], dataset)

In [25]:
from comment import CommentRepository
from db_config import SessionLocal
from dotenv import load_dotenv
import os

load_dotenv()
session = SessionLocal()
repo = CommentRepository()
iteration_limit = 0
while iteration_limit < 1:
    comments = repo.get_comments_with_no_prediction(limit=200)
    test_df = [{
        'id': comment.id,
        'text': comment.text,
        'label': None
    } for comment in comments]

    prediction_dataset = Dataset.from_pandas(df).train_test_split(test_size=1.0, seed=42)
    prediction_dataset = prediction_dataset.remove_columns(['__index_level_0__'])
    prediction_dataset['train'] = dataset['train']
    print(prediction_dataset)
    iteration_limit += 1
    print('anything..')
    test_y, pred_y = detect_satd(MODEL_CONFIGS[-1:], FEW_SHOT_SIZES[2:3], PROMPT_TEMPLATES[:1], FEW_SHOT_STRATEGIES[2:3], dataset)
    for _,id, pred, label in enumerate(zip(test_df['id'], test_df['label'], pred_y)):
        target_comment = repo.get_comment(id)
        target_comment.pred_td = True if label.lower() == 'yes' else False
        repo.merge_comments(target_comment)

ValueError: test_size=1.0 should be either positive and smaller than the number of samples 1006 or a float in the (0, 1) range